# Pydantic Essentials — Recap

Minimum you need: build a `BaseModel`, know required vs. optional fields, add simple validation, and convert back to dict/JSON. Everything after that is useful, not blocking.

## 1. The problem without Pydantic

In [1]:
# Plain Python: every function re-checks types and ranges by hand.
def add_order(product: str, quantity: int):
    if not isinstance(product, str) or not isinstance(quantity, int):
        raise TypeError("product must be str, quantity must be int")
    if quantity <= 0:
        raise ValueError("quantity must be positive")
    print(f"Order added: {quantity} x {product}")

add_order("keyboard", 3)

Order added: 3 x keyboard


## 2. The same thing with a `BaseModel`

In [2]:
from pydantic import BaseModel

class Order(BaseModel):
    product: str
    quantity: int

def add_order(order: Order):
    print(f"Order added: {order.quantity} x {order.product}")

order = Order(product="keyboard", quantity="3")  # "3" (str) is coerced to int
add_order(order)

Order added: 3 x keyboard


## 3. Required vs. optional fields

In [3]:
from typing import Optional, List

class User(BaseModel):
    username: str
    email: str
    age: Optional[int] = None
    tags: List[str] = []

alice = User(username="alice", email="alice@example.com")
print(alice)

username='alice' email='alice@example.com' age=None tags=[]


`username` and `email` are required — leave either out and Pydantic raises `ValidationError`. `age` and `tags` have defaults, so they're optional.

## 4. Nested models

In [4]:
class Address(BaseModel):
    city: str
    state: str
    pin: str

class Employee(BaseModel):
    name: str
    role: str
    address: Address

emp = Employee(
    name="Kishor",
    role="Engineer",
    address={"city": "Gurgaon", "state": "Haryana", "pin": "122001"},
)
print(emp.address.city)

Gurgaon


## 5. Field constraints & built-in types

`EmailStr` needs the extra: `uv add pydantic[email]` (or `email-validator`).

In [5]:
from pydantic import EmailStr, Field

class Customer(BaseModel):
    name: str = Field(max_length=50)
    email: EmailStr
    age: int = Field(gt=0, lt=120)
    rating: float = Field(ge=0, le=5)

customer = Customer(name="Bappy", email="bappy@example.com", age=28, rating=4.5)
print(customer)

name='Bappy' email='bappy@example.com' age=28 rating=4.5


## 6. `field_validator` — a custom rule on one field

In [6]:
from pydantic import field_validator

class SupportTicket(BaseModel):
    title: str
    priority: str

    @field_validator("priority")
    @classmethod
    def priority_must_be_known(cls, value: str) -> str:
        allowed = {"low", "medium", "high"}
        if value not in allowed:
            raise ValueError(f"priority must be one of {allowed}")
        return value

print(SupportTicket(title="Login broken", priority="high"))

try:
    SupportTicket(title="x", priority="urgent")
except Exception as e:
    print("rejected as expected:", e.errors()[0]["msg"])

title='Login broken' priority='high'
rejected as expected: Value error, priority must be one of {'low', 'high', 'medium'}


## 7. `model_validator` — a rule across multiple fields

In [7]:
from pydantic import model_validator

class Event(BaseModel):
    name: str
    start_hour: int
    end_hour: int

    @model_validator(mode="after")
    def end_after_start(self):
        if self.end_hour <= self.start_hour:
            raise ValueError("end_hour must be after start_hour")
        return self

print(Event(name="Standup", start_hour=9, end_hour=10))

try:
    Event(name="Bad", start_hour=10, end_hour=9)
except Exception as e:
    print("rejected as expected:", e.errors()[0]["msg"])

name='Standup' start_hour=9 end_hour=10
rejected as expected: Value error, end_hour must be after start_hour


## 8. `computed_field` — derived, read-only values

In [8]:
from pydantic import computed_field

class Rectangle(BaseModel):
    width: float
    height: float

    @computed_field
    @property
    def area(self) -> float:
        return round(self.width * self.height, 2)

box = Rectangle(width=3, height=4)
print(box.area)

12.0


## 9. Serialization — back to dict / JSON

In [9]:
print(box.model_dump())        # dict
print(box.model_dump_json())   # JSON string

{'width': 3.0, 'height': 4.0, 'area': 12.0}
{"width":3.0,"height":4.0,"area":12.0}


Pydantic **v2** only (per `AGENT_RULES.md`): `model_dump()` / `model_dump_json()`, never the old v1 `.dict()`.

## 10. Recap

- `BaseModel` is a typed contract — invalid data raises `ValidationError` instead of silently passing through.
- Optional fields need a default; required fields don't.
- Nested models compose exactly like any other field type.
- `field_validator` → one field. `model_validator` → rules across several fields.
- `model_dump()` / `model_dump_json()` are how a model becomes plain data again.

This is also exactly what `llm.with_structured_output(YourModel)` hands back in Modules 5, 11, 13, 14, 16 — a typed object instead of text you'd otherwise have to parse.